In [1]:
import pandas as pd

In [2]:
# Input
raw_nodes_csv = "breast_cancer/raw_nodes.csv"
augmented_csv = "breast_cancer/confidence_rank_augmented_temp.csv"

# Output
unique_edges_csv = "breast_cancer/confidence_rank_unique_temp.csv"
unique_nodes_csv = "breast_cancer/unique_nodes.csv"

In [3]:
df = pd.read_csv(augmented_csv, dtype=str)

In [4]:
# ------------------------------------------------------------------
# Remove rows marked as unsupported or replaced
# ------------------------------------------------------------------

df = df[df["Classification"] != "unsupported"].copy()

df = df[df["Classification"] != "replaced"].copy()

In [5]:
# ------------------------------------------------------------------
# Report ambiguous signs
# ------------------------------------------------------------------

ambiguous = df["Sign"] == "ambiguous"

if ambiguous.any():
    print("Rows with ambiguous Sign:\n")
    print(df.loc[ambiguous])
    print()

In [6]:
# ------------------------------------------------------------------
# Group by Regulator and Target
# ------------------------------------------------------------------

cols_that_should_match = [
    "Sign",
    "Confidence rank",
    "Constraint",
    "Classification",
]

rows = []

for (reg, tar), group in df.groupby(["Regulator", "Target"], sort=True):

    # Check consistency
    inconsistent = False
    for col in cols_that_should_match:
        values = group[col].fillna("").unique()
        if len(values) > 1:
            print(f"Inconsistency for edge {reg} -> {tar}")
            print(f"  {col}: {list(values)}")
            inconsistent = True

    if inconsistent:
        print("Inconsistent rows:\n")
        print(group)
        print()

    # Collect models
    models = sorted(group["Model"].dropna().unique())

    # Build merged row
    row = {
        "Regulator": reg,
        "Target": tar,
        "Sign": group.iloc[0]["Sign"],
        "Model": ", ".join(models),
        "Classification": group.iloc[0]["Classification"],
        "Confidence rank": group.iloc[0]["Confidence rank"],
        "Constraint": group.iloc[0]["Constraint"],
    }

    rows.append(row)

merged_df = pd.DataFrame(rows)

# ------------------------------------------------------------------
# Sort
# ------------------------------------------------------------------

merged_df = (
    merged_df
    .sort_values(by=list(merged_df.columns))
    .reset_index(drop=True)
)


In [7]:
print(f"Original rows : {len(df)}")
print(f"Merged rows   : {len(merged_df)}")

merged_df.head()

Original rows : 1887
Merged rows   : 700


,Regulator,Target,Sign,Model,Classification,Confidence rank,Constraint
0,ABL1,JAK2,positive,"TGEN, TM, TT",direct,1,regulates
1,ABL1,MYOD1,negative,"TGEN, TM",assumption,4,NaN
2,ABL1,SRC,positive,"TGEN, TM",assumption,4,NaN
3,ABL2,CEBPB,positive,"TGEN, TM, TT",direct,1,regulates
4,AKT,AKT,positive,"BL, BS, HL, HS, SL, SS",indirect,2,NaN


In [8]:
merged_df.to_csv(unique_edges_csv, index=False)

In [9]:
# ------------------------------------------------------------------
# Merge nodes
# ------------------------------------------------------------------

rows = []

# Combine Regulator and Target into a single node/model table
node_model_df = pd.concat(
    [
        df[["Regulator", "Model"]].rename(columns={"Regulator": "Node"}),
        df[["Target", "Model"]].rename(columns={"Target": "Node"}),
    ],
    ignore_index=True,
)

# Group by Node
for node, group in node_model_df.groupby("Node", sort=True):

    # Collect models in which this node appeared
    models = sorted(group["Model"].dropna().unique())

    rows.append({
        "Node": node,
        "Model": ", ".join(models),
    })

merged_nodes_df = pd.DataFrame(rows)

# ------------------------------------------------------------------
# Sort
# ------------------------------------------------------------------

merged_nodes_df = (
    merged_nodes_df
    .sort_values(by=["Node", "Model"])
    .reset_index(drop=True)
)

In [10]:
# ------------------------------------------------------------------
# Import BBM node mapping
# ------------------------------------------------------------------

bbm_df = pd.read_csv(raw_nodes_csv)

In [11]:
# ------------------------------------------------------------------
# Aggregate imported CSV by Our notation
# ------------------------------------------------------------------
# There can be multiple rows with the same Our notation.

bbm_grouped = (
    bbm_df
    .groupby("Our notation", sort=True)
    .agg({
        "BBM node": lambda x: ", ".join(sorted(x.dropna().astype(str).unique())),
        "Model": lambda x: ", ".join(sorted(x.dropna().astype(str).unique())),
        "Input in BBM": lambda x: ", ".join(sorted(x.dropna().astype(str).unique())),
        "Input in merge": lambda x: ", ".join(sorted(x.dropna().astype(str).unique())),
    })
    .reset_index()
)


In [12]:
# ------------------------------------------------------------------
# Sanity check: nodes must exist in both dataframes
# ------------------------------------------------------------------

merged_nodes = set(merged_nodes_df["Node"].dropna())
csv_nodes = set(bbm_grouped["Our notation"].dropna())

# Nodes present in merged_nodes_df but missing from the CSV
only_in_merged = sorted(merged_nodes - csv_nodes)

# Nodes present in CSV but missing from merged_nodes_df
only_in_csv = sorted(csv_nodes - merged_nodes)

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------

print("Number of nodes in merged_nodes_df:", len(merged_nodes))
print("Number of nodes in CSV:", len(csv_nodes))
print()

if only_in_merged:
    print("Nodes only in merged_nodes_df:")
    for node in only_in_merged:
        print(f"  {node}")
else:
    print("No nodes exist only in merged_nodes_df.")

print()

if only_in_csv:
    print("Nodes only in CSV:")
    for node in only_in_csv:
        print(f"  {node}")
else:
    print("No nodes exist only in CSV.")

Number of nodes in merged_nodes_df: 200
Number of nodes in CSV: 198

Nodes only in merged_nodes_df:
  ERK_2
  MEK_2
  RAF1_2

Nodes only in CSV:
  RND1


In [13]:
# ------------------------------------------------------------------
# Merge
# ------------------------------------------------------------------

merged_nodes_df = merged_nodes_df.merge(
    bbm_grouped[
        [
            "Our notation",
            "BBM node",
            "Model",
            "Input in BBM",
            "Input in merge",
        ]
    ].rename(columns={"Model": "BBM models"}),
    how="outer",
    left_on="Node",
    right_on="Our notation",
)

# Rename the existing Model column
merged_nodes_df = merged_nodes_df.rename(
    columns={"Model": "Corrected models"}
)

# Our notation is only needed for matching, then sort and index
merged_nodes_df = (
    merged_nodes_df
    .drop(columns=["Our notation"])
    .sort_values(by="Node")
    .reset_index(drop=True)
)

# Add index as the first column
merged_nodes_df.insert(0, "index", merged_nodes_df.index + 1)

In [14]:
merged_nodes_df.to_csv(unique_nodes_csv, index=False)

print(f"Merged rows   : {len(merged_nodes_df)}")

Merged rows   : 201
